In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
from copy import deepcopy
from functools import partial
from itertools import combinations

# Import sklearn classes for model selection, cross validation, and performance evaluation
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, accuracy_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from category_encoders import OneHotEncoder, OrdinalEncoder, CountEncoder

# Import libraries for Hypertuning
import optuna

# Import libraries for gradient boosting
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoost, CatBoostRegressor, CatBoostClassifier
from catboost import Pool

# Suppress warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

### Load Data

In [ ]:
df_train = pd.read_csv('/kaggle/input/playground-series-s3e8/train.csv',index_col='id')
df_train['is_original']= False
df_test = pd.read_csv('/kaggle/input/playground-series-s3e8/test.csv',index_col='id')
df_test['is_original']= False

In [ ]:
original = pd.read_csv('/kaggle/input/gemstone-price-prediction/cubic_zirconia.csv',index_col=[0])

# depth has missing values
original = original[-original.depth.isna()]
df_train = pd.concat([df_train, original]).drop_duplicates()

In [ ]:
# Drop the columns 'price' and 'id' from the training data
target_col = 'price'
X_train = df_train.drop([f'{target_col}'],axis=1).reset_index(drop=True)
y_train = df_train[f'{target_col}'].reset_index(drop=True)
X_test = df_test.reset_index(drop=True)

print(f"X_train shape :{X_train.shape} , y_train shape :{y_train.shape}")
print(f"X_test shape :{X_test.shape}")

del df_train, df_test

### Aggregate Featrues

In [ ]:
class AggFeatureExtractor(BaseEstimator, TransformerMixin):
    
    def __init__(self, group_col, agg_col, agg_func):
        self.group_col = group_col
        self.group_col_name = ''
        for col in group_col:
            self.group_col_name += col
        self.agg_col = agg_col
        self.agg_func = agg_func
        self.agg_df = None
        self.medians = None
        
    def fit(self, X, y=None):
        group_col = self.group_col
        agg_col = self.agg_col
        agg_func = self.agg_func
        
        self.agg_df = X.groupby(group_col)[agg_col].agg(agg_func)
        self.agg_df.columns = [f'{self.group_col_name}_{agg}_{_agg_col}' for _agg_col in agg_col for agg in agg_func]
        self.medians = X[agg_col].median()
        
        return self
    
    def transform(self, X):
        group_col = self.group_col
        agg_col = self.agg_col
        agg_func = self.agg_func
        agg_df = self.agg_df
        medians = self.medians
        
        X_merged = pd.merge(X, agg_df, left_on=group_col, right_index=True, how='left')
        X_merged.fillna(medians, inplace=True)
        X_agg = X_merged.loc[:, [f'{self.group_col_name}_{agg}_{_agg_col}' for _agg_col in agg_col for agg in agg_func]]
        
        return X_agg
    
    def fit_transform(self, X, y=None):
        self.fit(X, y)
        X_agg = self.transform(X)
        return X_agg

In [ ]:
class Preprocessor:
    def __init__(self, agg_col, agg_func, group_cols, comb_cat_cols, oh_cat_cols, ce_cat_cols):
        self.agg_col = agg_col
        self.agg_func = agg_func
        self.group_cols = group_cols
        self.comb_cat_cols = comb_cat_cols
        self.oh_cat_cols = oh_cat_cols
        self.ce_cat_cols = ce_cat_cols
        
    def preprocess(self, X_train, X_test):
        X_train = self.create_feature(X_train)
        X_test = self.create_feature(X_test)
        
        agg_train, agg_test = [], []
        for group_col in self.group_cols:
            agg_extractor = AggFeatureExtractor(group_col=group_col, agg_col=self.agg_col, agg_func=self.agg_func)
            agg_extractor.fit(pd.concat([X_train, X_test], axis=0))
            agg_train.append(agg_extractor.transform(X_train))
            agg_test.append(agg_extractor.transform(X_test))
        X_train = pd.concat([X_train] + agg_train, axis=1)
        X_test = pd.concat([X_test] + agg_test, axis=1)

        # create_categorical_combinations
        X_train, _ = self.create_categorical_combinations(X_train, self.comb_cat_cols)
        X_test, _ = self.create_categorical_combinations(X_test, self.comb_cat_cols)
        
        # OneHotEncoder
        onehot_train, onehot_test = self.encode_categorical_features(X_train, X_test, self.oh_cat_cols, encoder_type='onehot')

        # CountEncoder
        count_train, count_test = self.encode_categorical_features(X_train, X_test, self.ce_cat_cols, encoder_type='count')

        X_train = pd.concat([X_train, onehot_train, count_train], axis=1).drop(self.ce_cat_cols, axis=1)
        X_test = pd.concat([X_test, onehot_test, count_test], axis=1).drop(self.ce_cat_cols, axis=1)
        
        return X_train, X_test
        
    def create_feature(self, df):
        df['y_carat'] = df['y'] * df['carat']
        df['x_carat'] = df['x'] * df['carat']
        df['area'] = df['y'] * df['x']
        df['area_carat'] = df['carat'] / (df['area'] + 1e-6)
        df['volume'] = df['x'] * df['y'] * df['z']
        df['density'] = df['carat'] / (df['volume'] + 1e-6)
        df['depth_per_volume'] = df['depth'] / (df['volume'] + 1e-6)
        df['depth_per_density'] = df['depth'] / (df['density'] + 1e-6)
        df['depth_per_table'] = df['depth'] / (df['table'] + 1e-6)
        return df

    def create_categorical_combinations(self, df, categorical_columns, max_pattern=3):
        cols = []
        for comb in range(len(categorical_columns)):
            for col in combinations(categorical_columns, comb+1):
                if len(list(col)) > max_pattern:
                    break
                if len(list(col)) > 1:
                    col_names = list(col)
                    new_col = '_'.join(col_names)
                    df[new_col] = df[col_names[0]].astype(str)
                    for c in col_names[1:]:
                        df[new_col] = df[new_col] + '_' + df[c].astype(str)
                cols.append('_'.join(col))
        return df, cols
    
    def encode_categorical_features(self, X_train, X_test, categorical_columns, encoder_type):
        if encoder_type == 'onehot':
            encoder = OneHotEncoder(cols=categorical_columns)
            train_encoder = encoder.fit_transform(X_train[categorical_columns]).add_suffix('_ohe')
            test_encoder = encoder.transform(X_test[categorical_columns]).add_suffix('_ohe')
        elif encoder_type == 'count':
            encoder = CountEncoder(cols=categorical_columns)
            train_encoder = encoder.fit_transform(X_train[categorical_columns]).add_suffix('_count')
            test_encoder = encoder.transform(X_test[categorical_columns]).add_suffix('_count')
        else:
            raise ValueError("Unsupported encoder type. Available options: 'onehot', 'count'")

        return train_encoder, test_encoder

In [ ]:
%%time
agg_col = ['area_carat', 'volume', 'density', 'depth_per_volume', 'depth_per_density', 'depth_per_table']
agg_func = ['mean', 'std']
group_cols = [['cut'], ['color'], ['clarity'], ['cut', 'color'], ['cut', 'clarity'], ['color', 'clarity'], ['cut', 'color', 'clarity']]
comb_categorical_columns = ['cut', 'color', 'clarity']
oh_categorical_columns = ['cut', 'color', 'clarity', 'is_original']
ce_categorical_columns = ['cut', 'color', 'clarity', 'is_original', 'cut_color', 'cut_clarity', 'color_clarity', 'cut_color_clarity']

pp = Preprocessor(agg_col, agg_func, group_cols, comb_categorical_columns, oh_categorical_columns, ce_categorical_columns)
X_train, X_test = pp.preprocess(X_train, X_test)

print(f"X_train shape :{X_train.shape} , y_train shape :{y_train.shape}")
print(f"X_test shape :{X_test.shape}")

### Split Data

In [ ]:
class Splitter:
    def __init__(self, test_size=0.2, kfold=True, n_splits=3):
        self.test_size = test_size
        self.kfold = kfold
        self.n_splits = n_splits

    def split_data(self, X, y, random_state_list):
        if self.kfold:
            for random_state in random_state_list:
                kf = KFold(n_splits=self.n_splits, random_state=random_state, shuffle=True)
                for train_index, val_index in kf.split(X, y):
                    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
                    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
                    yield X_train, X_val, y_train, y_val
        else:
            for random_state in random_state_list:
                X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=self.test_size, random_state=random_state)
                yield X_train, X_val, y_train, y_val

kfold = True
n_splits = 5
random_state = 2023
random_state_list = [_ for _ in range(6)] # split_data [42, 71, 25]

splitter = Splitter(kfold=kfold, n_splits=n_splits)

### Set Hyperparameters

In [ ]:
# Hyperparameters
n_estimators = 9999
device = "cpu"

xgb_params = {
    'n_estimators': n_estimators,
    'learning_rate': 0.05,
    'max_depth': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror'
}

lgb_params = {
    'n_estimators': n_estimators,
    'learning_rate': 0.05,
    'max_depth': 10,
    'subsample_for_bin': 20000,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'regression',
    'device': device
}

cb_params = {
    'n_estimators': n_estimators,
    'learning_rate': 0.05,
    'max_depth': 8,
    'loss_function': 'RMSE',
    'task_type': device.upper()
}

### Model

In [ ]:
def train_regressor(regressor, X_train, y_train, X_val, y_val, regressor_params, early_stopping_rounds=500, random_state=42):
    regressor = regressor(**regressor_params, random_state=random_state)
    eval_set = [(X_val, y_val)]
    regressor.fit(X_train, y_train, early_stopping_rounds=early_stopping_rounds, eval_set=eval_set, verbose=1000)
    val_preds = regressor.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    return regressor, rmse

In [ ]:
xgb_models, lgb_models, cb_models = [], [], []
xgb_scores, lgb_scores, cb_scores = [], [], []
for X_train_, X_val, y_train_, y_val in splitter.split_data(X_train, y_train, random_state_list=random_state_list):
    xgb_model, xgb_rmse = train_regressor(xgb.XGBRegressor, X_train_, y_train_, X_val, y_val, xgb_params)
    lgb_model, lgb_rmse = train_regressor(lgb.LGBMRegressor, X_train_, y_train_, X_val, y_val, lgb_params)
    cb_model, cb_rmse = train_regressor(CatBoostRegressor, X_train_, y_train_, X_val, y_val, cb_params)
    xgb_models.append(deepcopy(xgb_model)), lgb_models.append(deepcopy(lgb_model)), cb_models.append(deepcopy(cb_model))
    xgb_scores.append(xgb_rmse), lgb_scores.append(lgb_rmse), cb_scores.append(cb_rmse)

### Ensemble

In [ ]:
class OptunaWeights:
    def __init__(self, random_state):
        self.study = None
        self.weights = None
        self.random_state = random_state

    def _objective(self, trial, y_true, y_preds):
        # Define the weights for the predictions from each model
        weights = [trial.suggest_float(f"weight{n}", 0, 1) for n in range(len(y_preds))]

        # Calculate the weighted prediction
        weighted_pred = np.average(np.array(y_preds).T, axis=1, weights=weights)

        # Calculate the ROC AUC score for the weighted prediction
        score = np.sqrt(mean_squared_error(y_true, weighted_pred))
        return score

    def fit(self, y_true, y_preds, n_trials=300):
        optuna.logging.set_verbosity(optuna.logging.ERROR)
        sampler = optuna.samplers.CmaEsSampler(seed=self.random_state)
        self.study = optuna.create_study(sampler=sampler, study_name="OptunaWeights", direction='minimize')
        objective_partial = partial(self._objective, y_true=y_true, y_preds=y_preds)
        self.study.optimize(objective_partial, n_trials=n_trials)
        self.weights = [self.study.best_params[f"weight{n}"] for n in range(len(y_preds))]

    def predict(self, y_preds):
        assert self.weights is not None, 'OptunaWeights error, must be fitted before predict'
        weighted_pred = np.average(np.array(y_preds).T, axis=1, weights=self.weights)
        return weighted_pred

    def fit_predict(self, y_true, y_preds, n_trials=300):
        self.fit(y_true, y_preds, n_trials=n_trials)
        return self.predict(y_preds)
    
    def weights(self):
        return self.weights

In [ ]:
models = np.array([xgb_models, lgb_models, cb_models]).T.tolist()
names = ['XGBoost', 'LightGBM', 'CatBoost']

# Initialize variables
test_predss = np.zeros(X_test.shape[0])
ensemble_score = []
weights = []

# Loop through the folds
i = 0
for _model, (X_train_, X_val, y_train_, y_val) in zip(models, splitter.split_data(X_train, y_train, random_state_list=random_state_list)):
    n = i % n_splits
    m = i // n_splits
    val_preds, test_preds, zero_test_preds = [], [], []
    for model in _model:
        oof_pred = model.predict(X_val.values)
        test_pred = model.predict(X_test.values)
        val_preds.append(oof_pred)
        test_preds.append(test_pred)
        
    # Use Optuna to find the best ensemble weights
    optweights = OptunaWeights(random_state=random_state)
    val_pred = optweights.fit_predict(y_val.values, val_preds)
    score = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f'[FOLD-{n} SEED-{random_state_list[m]}] RMSE score {score:.5f}')
    ensemble_score.append(score)
    weights.append(optweights.weights)
    
    # Predict on the test set using the optimized ensemble weights
    test_predss += optweights.predict(test_preds) / (n_splits * len(random_state_list))
    i += 1
    
# Calculate the mean AUC score of the ensemble
mean_score = np.mean(ensemble_score)
std_score = np.std(ensemble_score)
print(f'Ensemble RMSE score {mean_score:.5f} ± {std_score:.5f}')

# Print the mean and standard deviation of the ensemble weights for each model
print('--- Model Weights ---')
mean_weights = np.mean(weights, axis=0)
std_weights = np.std(weights, axis=0)
for name, mean_weight, std_weight in zip(names, mean_weights, std_weights):
    print(f'{name} {mean_weight:.5f} ± {std_weight:.5f}')

In [ ]:
def visualize_importance(models, feature_cols, title, top=20):
    importances = []
    feature_importance = pd.DataFrame()
    for i, model in enumerate(models):
        _df = pd.DataFrame()
        _df["importance"] = model.feature_importances_
        _df["feature"] = pd.Series(feature_cols)
        _df["fold"] = i
        _df = _df.sort_values('importance', ascending=False)
        _df = _df.head(top)
        feature_importance = pd.concat([feature_importance, _df], axis=0, ignore_index=True)
        
    feature_importance = feature_importance.sort_values('importance', ascending=False)
    plt.figure(figsize=(10, 5))
    sns.barplot(x='importance', y='feature', data=feature_importance, color='skyblue', errorbar='sd')
    plt.xlabel('Importance', fontsize=14)
    plt.ylabel('Feature', fontsize=14)
    plt.title(f'{title} Feature Importance [Top {top}]', fontsize=18)
    plt.grid(True, axis='x')
    plt.show()
    
visualize_importance(xgb_models, list(X_train.columns), 'XGBoost Top')
visualize_importance(lgb_models, list(X_train.columns), 'LightGBM')
visualize_importance(cb_models, list(X_train.columns), 'CatBoost')

### Make Submission

In [ ]:
submission_df = pd.read_csv("/kaggle/input/playground-series-s3e8/sample_submission.csv", index_col='id')
submission_df['price'] = test_predss
submission_df.to_csv('Ensemble_submission.csv')

In [ ]:
sns.histplot(submission_df.price)